# 시계열 분석

**시계열**은 시간 순서에 따라 기록한 관측값들의 모음입니다. 앞 장의 회귀도 사용할 수 있지만, 시간 순서가 있는 자료에 특히 유용한 방법들이 더 있습니다.

2001~2024년 미국의 발전량과 같은 기간의 기상 자료를 살펴보겠습니다. 시계열을 장기 추세와 반복되는 계절 성분으로 나누고, 회귀로 추세를 적합해 예측한 뒤, **자기회귀 누적 이동평균**이라는 모형을 사용하겠습니다. 긴 이름 대신 보통 ARIMA라고 부릅니다.

> 번역자 주: 원문 도입부는 재생에너지 발전량이라고 표현하지만, 첫 예제의 원자력은 재생에너지가 아닙니다. 자료에는 여러 발전원이 들어 있으며 이 장은 원자력과 태양광을 다룹니다.

《Think Stats》 3판은 [Bookshop.org](https://bookshop.org/a/98697/9781098190255)와 [Amazon](https://amzn.to/42lmxwu)에서 구할 수 있습니다. 두 링크는 원저자의 제휴 링크입니다. 책이 도움이 되었다면 Green Tea Press의 [무료 공개 교육 자료 제작을 후원](https://greenteapress.com/support/?book=ts3)하는 것도 생각해 주세요.

[원문 노트북을 Colab에서 실행하기](https://colab.research.google.com/github/AllenDowney/ThinkStats/blob/30ecbdcf05723f1190cae2aeaff5848f59f8ca26/nb/chap12.ipynb).

In [1]:
from os.path import basename, exists


def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + local)


download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/thinkstats.py")

In [2]:
try:
    import empiricaldist
except ImportError:
    %pip install empiricaldist

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from thinkstats import decorate

plt.rcParams["figure.dpi"] = 300

## 발전량

미국 에너지정보청(EIA)의 월별 발전량 자료를 사용합니다. 2001~2024년의 여러 발전원 자료를 포함하며, 내려받는 방법은 이 장의 노트북에 있습니다.

원저자는 [EIA 전력 자료 브라우저](https://www.eia.gov/electricity/data/browser/)에서 2024년 9월 17일에 자료를 내려받았습니다. 아래 예제의 2024년 수치는 당시 자료에 포함된 기간까지입니다.

In [4]:
filename = "Net_generation_for_all_sectors.csv"
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/" + filename)

읽은 표를 분석하기 편한 형태로 바꾸겠습니다.

In [5]:
elec = (
    pd.read_csv("Net_generation_for_all_sectors.csv", skiprows=4)
    .drop(columns=["units", "source key"])
    .set_index("description")
    .replace("--", np.nan)
    .transpose()
    .astype(float)
)

정리한 표의 각 열은 한 발전원의 월별 총발전량이며 단위는 기가와트시(GWh)입니다. 열 이름은 다음과 같습니다.

In [6]:
elec.columns

인덱스에는 연월을 나타내는 문자열이 있습니다. 처음 12개를 보겠습니다.

In [7]:
elec.index[:12]

문자열을 pandas의 `Timestamp`로 바꾸면 다루기 쉽습니다. `date_range`를 2001년 1월부터 빈도 코드 `ME`, 즉 월말로 지정하면 매달 마지막 날이 생성됩니다.

In [8]:
elec.index = pd.date_range(start="2001-01", periods=len(elec), freq="ME")
elec.index[:6]

인덱스는 이제 `DatetimeIndex`입니다. NumPy의 `datetime64[ns]`는 날짜마다 64비트를 쓰고 나노초 정밀도를 지원한다는 뜻입니다.

## 시계열 분해

먼저 2001년 1월~2024년 6월의 미국 원자력 발전량을 장기 추세와 주기 성분으로 나누겠습니다. 월별 발전량은 다음과 같습니다.

In [9]:
actual_options = dict(color="C0", lw=1, alpha=0.6)
trend_options = dict(color="C1", alpha=0.6)
pred_options = dict(color="C2", alpha=0.6, ls=':')
model_options = dict(color="gray", alpha=0.6, ls='--')

In [10]:
nuclear = elec["United States : nuclear"]
nuclear.plot(label="nuclear", **actual_options)

decorate(ylabel="GWh")

오르내리는 추세가 있지만 월별 변동이 커서 보기 어렵습니다. `rolling`과 `mean`으로 **이동평균**을 구하면 장기 흐름을 더 잘 볼 수 있습니다.

In [11]:
trend = nuclear.rolling(window=12).mean()

`window=12`는 겹치는 12개월 구간을 뜻합니다. 첫 구간은 첫 측정부터 12개, 다음 구간은 두 번째부터 12개를 담고 각각의 평균을 구합니다. 원자료와 함께 그리겠습니다.

In [12]:
nuclear.plot(label="nuclear", **actual_options)
trend.plot(label="trend", **trend_options)
decorate(ylabel="GWh")

추세에도 변동이 남습니다. 더 긴 창을 쓰면 더 매끄러워지겠지만 여기서는 12개월을 유지합니다. 원자료에서 추세를 빼면 장기적인 평균 변화가 줄어든 시계열이 나옵니다.

In [13]:
detrended = (nuclear - trend).dropna()
detrended.plot(label="detrended", **actual_options)
decorate(ylabel="GWh")

해마다 반복되는 패턴이 보입니다. 겨울 난방과 여름 냉방처럼 전력 수요가 계절에 따라 달라지기 때문일 수 있습니다. 날짜의 월을 기준으로 묶어 추세를 뺀 값의 월별 평균을 구합니다.

In [14]:
monthly_averages = detrended.groupby(detrended.index.month).mean()
monthly_averages.plot(label="monthly average", **actual_options)
decorate(ylabel="GWh")

1은 1월, 12는 12월입니다. 추운 달과 더운 달에 발전량이 높고 4월과 10월에 낮습니다. 각 날짜의 월에 해당하는 평균을 찾아 원자료와 같은 길이의 계절 성분을 만듭니다.

In [15]:
seasonal = monthly_averages[nuclear.index.month]
seasonal.index = nuclear.index
seasonal.plot(label="seasonal", **actual_options)
decorate(ylabel="GWh")

12개월 패턴이 똑같이 반복됩니다. 추세와 계절 성분을 더하면 각 달의 기대값을 나타낼 수 있습니다.

In [16]:
expected = trend + seasonal

그 합을 원자료와 비교하겠습니다.

In [17]:
expected.plot(label="expected", **pred_options)
nuclear.plot(label="actual", **actual_options)
decorate(ylabel="GWh")

원자료에서 추세와 계절 성분의 합을 빼면 기대값에서 벗어난 **잔차 성분**이 남습니다.

In [18]:
resid = nuclear - expected
resid.plot(label="residual", **actual_options)
decorate(ylabel="GWh")

잔차에는 추세와 계절 패턴이 설명하지 못한 날씨, 설비 정비, 특정 사건에 따른 수요 변화 등이 섞입니다. 예측하기 어렵거나 알 수 없는 기여가 많아 이를 확률적인 양으로 취급하곤 합니다. 잔차 분포를 그려 보겠습니다.

In [19]:
from thinkstats import plot_kde

plot_kde(resid.dropna())
decorate(xlabel="Residual (GWh)", ylabel="Density")

정규분포의 종 모양과 비슷합니다. 여러 무작위 기여가 더해진다는 설명과 부합하지만, 모양만으로 독립성이나 정규성이 입증되는 것은 아닙니다. 원자료 분산과 잔차 분산을 비교해 적합 정도를 구하겠습니다.

In [20]:
rsquared = 1 - resid.var() / nuclear.var()
rsquared

$R^2$는 약 0.905입니다. 추세와 계절 성분이 원자료 변동의 약 90.5%를 담고 있습니다. 앞 장보다 높지만, 이 값은 분해에 사용한 자료를 얼마나 잘 설명하는지 나타냅니다. 미래 예측의 정확도는 따로 확인해야 합니다.

이 과정을 **계절 분해**라고 합니다. statsmodels의 `seasonal_decompose`로도 계산할 수 있습니다.

> 번역자 주: 원문 설명의 약 0.92는 바로 앞 코드 출력 0.905456과 달라, 출력에 맞춰 고쳤습니다. 뒤의 `seasonal_decompose`는 중심 이동평균을 사용하므로 앞의 후행 이동평균을 쓴 수동 분해와 결과가 다를 수 있습니다.

In [21]:
from statsmodels.tsa.seasonal import seasonal_decompose

decomposition = seasonal_decompose(nuclear, model="additive", period=12)

`model="additive"`는 추세·계절·잔차를 더하는 가법 모형입니다. `period=12`는 계절 주기가 12개월이라는 뜻입니다. 반환 객체에 세 성분이 들어 있으며, 다음 함수로 그릴 수 있습니다.

In [22]:
def plot_decomposition(original, decomposition):
    plt.figure(figsize=(6, 5))

    ax1 = plt.subplot(4, 1, 1)
    plt.plot(original, label="Original", color="C0", lw=1)
    plt.ylabel("Original")

    plt.subplot(4, 1, 2, sharex=ax1)
    plt.plot(decomposition.trend, label="Trend", color="C1", lw=1)
    plt.ylabel("Trend")

    plt.subplot(4, 1, 3, sharex=ax1)
    plt.plot(decomposition.seasonal, label="Seasonal", color="C2", lw=1)
    plt.ylabel("Seasonal")

    plt.subplot(4, 1, 4, sharex=ax1)
    plt.plot(decomposition.resid, label="Residual", color="C3", lw=1)
    plt.ylabel("Residual")

    plt.tight_layout()

In [23]:
plot_decomposition(nuclear, decomposition)

직접 계산한 결과와 비슷하고, 세부 구현에 따른 차이가 있습니다. 계절 분해는 구조를 이해하는 데 도움이 되며 예측에도 사용할 수 있습니다.

## 예측

모형을 맞추는 **훈련 시계열**과, 그 모형의 예측 정확도를 확인하는 **평가 시계열**로 자료를 나누겠습니다.

In [24]:
def split_series(series, n=60):
    training = series.iloc[:-n]
    test = series.iloc[-n:]
    return training, test

평가 구간을 60개월로 정하면 2019년 7월부터 5년입니다.

In [25]:
training, test = split_series(nuclear)
test.index[0]

2019년 6월 시점에서 향후 5년을 예측한다고 생각해 보세요. 그때까지 관측된 훈련 자료만 계절 분해합니다.

In [26]:
decomposition = seasonal_decompose(training, model="additive", period=12)
trend = decomposition.trend

추세에 선형 모형을 맞춥니다. 설명변수 `months`는 자료 시작부터 지난 개월 수입니다.

In [27]:
import statsmodels.formula.api as smf

months = np.arange(len(trend))
data = pd.DataFrame({"trend": trend, "months": months}).dropna()
results = smf.ols("trend ~ months", data=data).fit()

회귀 결과를 확인합니다.

In [28]:
from thinkstats import display_summary

display_summary(results)

추세에 대한 선형 모형의 $R^2$는 약 0.35로 아주 높지는 않습니다. 훈련 기간과 평가 기간의 기대 추세를 계산해 그림으로 살펴보겠습니다.

In [29]:
months = np.arange(len(training) + len(test))
df = pd.DataFrame({"months": months})
pred_trend = results.predict(df)
pred_trend.index = nuclear.index

분해한 추세와 선형 모형을 함께 그립니다.

In [30]:
trend.plot(**trend_options)
pred_trend.plot(label="linear model", **model_options)
decorate(ylabel="GWh")

직선이 담지 못하는 변화가 있지만 전반적으로 증가하는 경향이 보입니다. 계절 성분을 월별 평균으로 정리합니다.

In [31]:
seasonal = decomposition.seasonal
monthly_averages = seasonal.groupby(seasonal.index.month).mean()

예측할 날짜의 월을 이 월별 평균에 대응시키면 계절 성분의 예측이 나옵니다.

In [32]:
pred_seasonal = monthly_averages[pred_trend.index.month]
pred_seasonal.index = pred_trend.index

추세와 계절 성분을 더해 최종 예측을 구합니다.

In [33]:
pred = pred_trend + pred_seasonal

훈련 자료와 예측값을 그립니다.

In [34]:
pred.plot(label="prediction", **pred_options)
training.plot(label="training", **actual_options)
decorate(ylabel="GWh")

훈련 자료에는 비교적 잘 맞고, 장기 추세가 계속된다는 가정 아래 미래 예측도 그럴듯합니다. 실제로 어떻게 되었는지 2019년 7월 이후의 평가 자료와 비교하겠습니다.

In [35]:
forecast = pred[test.index]
forecast.plot(label="predicted", **pred_options)
test.plot(label="actual", **actual_options)
decorate(ylabel="GWh")

첫해는 비교적 잘 맞았지만 2020년 원자력 발전량은 예측보다 낮았고, 이후에도 이전 추세로 돌아오지 않았습니다. 원저자는 코로나19의 영향을 가능성으로 언급합니다. 정확도를 수치화하기 위해 **평균절대백분율오차**(MAPE)를 사용하겠습니다.

In [36]:
def MAPE(predicted, actual):
    ape = np.abs(predicted - actual) / actual
    return np.mean(ape) * 100

이 예에서 예측의 평균절대백분율오차는 약 3.81%입니다.

In [37]:
MAPE(forecast, test)

뒤에서 다른 모형으로 돌아와 더 나은지 비교하겠습니다.

> 번역자 주: 시간 순서를 무작위로 섞어 훈련·평가 자료를 나누면 미래 정보가 새어 들어갈 수 있습니다. 분해와 추세 적합도 훈련 구간에서만 해야 합니다. 중심 이동평균의 과거 적합값에는 그 시점 이후의 자료가 쓰일 수 있어 실시간 예측값과 다릅니다. MAPE는 실제 값이 0이면 정의되지 않고 0에 가까울 때도 불안정합니다.

## 승법 모형

가법 모형은

$$y_t=T_t+S_t+e_t$$

처럼 추세·계절·잔차를 더합니다. 이 설명에서는 계절 변동과 잔차의 크기가 추세 수준에 비례해 커지지 않습니다. 2014년 이후 소규모 태양광 발전량은 이 가정에 잘 맞지 않습니다.

In [38]:
solar = elec["United States : small-scale solar photovoltaic"].dropna()
solar.plot(label="solar", **actual_options)
decorate(ylabel="GWh")

이 기간 발전량이 여러 배 커졌고 계절 변동의 폭도 커졌습니다. 계절과 무작위 변동의 크기가 추세에 비례한다면, 세 성분을 곱하는 승법 모형이 더 적절할 수 있습니다. 이 시계열도 훈련·평가 구간으로 나눕니다.

In [39]:
training, test = split_series(solar)

`model="multiplicative"`로 계절 분해합니다.

In [40]:
decomposition = seasonal_decompose(training, model="multiplicative", period=12)

분해한 결과를 그립니다.

In [41]:
plot_decomposition(training, decomposition)

계절 성분과 잔차 성분은 이제 더하는 양이 아니라 곱하는 배율입니다. 계절 성분은 추세의 약 0.68~1.26배 사이에서 움직입니다. 잔차 성분은 초기의 일부 예외를 빼면 대체로 1에서 ±5% 이내입니다. 각 성분을 꺼냅니다.

> 번역자 주: 원문의 ‘약 ±25%’는 위 분해의 실제 계절 배율 범위에 맞춰 고쳤습니다.

In [42]:
trend = decomposition.trend
seasonal = decomposition.seasonal
resid = decomposition.resid

이 모형의 훈련 자료 $R^2$는 매우 높습니다.

In [43]:
rsquared = 1 - resid.var() / training.var()
rsquared

태양광 발전량이 일조량을 따라 뚜렷한 연간 주기를 보이는 것은 자연스럽습니다. 장기 추세에는 이차 모형을 사용하겠습니다.

In [44]:
months = range(len(training))
data = pd.DataFrame({"trend": trend, "months": months}).dropna()
results = smf.ols("trend ~ months + I(months**2)", data=data).fit()

Patsy 수식의 `I(months**2)`는 제곱항을 추가하므로 별도 열을 만들 필요가 없습니다. 적합 결과를 보겠습니다.

In [45]:
display_summary(results)

일차항과 이차항의 p값은 작고 $R^2$는 높습니다. 과거와 미래에 대응하는 추세 예측값을 구합니다.

In [46]:
months = range(len(solar))
df = pd.DataFrame({"months": months})
pred_trend = results.predict(df)
pred_trend.index = solar.index

이차 추세를 그립니다.

In [47]:
pred_trend.plot(label="quadratic model", **model_options)
trend.plot(**trend_options)
decorate(ylabel="GWh")

과거 추세에 잘 맞습니다. 계절 성분으로 미래의 계절 패턴을 구하겠습니다.

In [48]:
monthly_averages = seasonal.groupby(seasonal.index.month).mean()
pred_seasonal = monthly_averages[pred_trend.index.month]
pred_seasonal.index = pred_trend.index

과거에 대한 **과거예측**(retrodiction)과 미래 예측을 만들기 위해 추세와 계절 성분을 곱합니다.

In [49]:
pred = pred_trend * pred_seasonal

훈련 자료와 함께 그린 결과입니다.

In [50]:
training.plot(label="training", **actual_options)
pred.plot(label="prediction", **pred_options)
decorate(ylabel="GWh")

과거 자료에 잘 맞고 미래 예측도 그럴듯합니다. 평가 자료와 비교해 실제 정확도를 보겠습니다.

In [51]:
future = pred[test.index]
future.plot(label="prediction", **pred_options)
test.plot(label="actual", **actual_options)
decorate(ylabel="GWh")

처음 3년은 잘 맞지만, 그 뒤에는 실제 성장이 예측을 웃돈 것으로 보입니다. 계절 분해는 이 태양광 예제에서 유용했지만, 원자력 예제에서는 덜 효과적이었습니다. 다음에는 자기회귀를 사용하겠습니다.

> 번역자 주: 큰 훈련 $R^2$와 작은 p값이 미래 추세의 지속을 보장하지는 않습니다. 시계열 잔차가 자기상관을 가지면 보통 OLS의 독립 오차를 전제로 한 표준오차와 p값도 주의해야 합니다. 승법 분해에는 양수 자료가 필요합니다.

## 자기회귀

자기회귀는 과거 값에 미래 값을 예측할 정보가 있다고 봅니다. 연간 주기가 있다면 다음 6월의 예측은 지난 6월 값에서 시작할 수 있습니다.

원자력 발전량에서 전년 같은 달과의 차이, 즉 **전년동월 차분**을 구하겠습니다.

In [52]:
diff = (nuclear - nuclear.shift(12)).dropna()
diff.plot(label="year over year differences", **actual_options)
decorate(ylabel="GWh")

차분의 크기는 원래 발전량보다 훨씬 작습니다. 원래 값보다 차분을 예측하기가 쉬울 수 있다는 것이 두 번째 생각입니다. 차분의 연속된 값들에 상관이 있으면 이를 예측에 쓸 수 있습니다. 각 달의 차분과 그보다 1·2·3개월 전의 차분을 한 표에 담습니다. 늦춘 열 이름이 `lag1`, `lag2`, `lag3`입니다.

In [53]:
df_ar = pd.DataFrame({"diff": diff})
for lag in [1, 2, 3]:
    df_ar[f"lag{lag}"] = diff.shift(lag)

df_ar = df_ar.dropna()

열 사이의 상관을 구합니다.

In [54]:
df_ar.corr()[["diff"]]

자기 자신을 시차를 두고 비교한 상관을 **자기상관**이라고 합니다. 인접한 두 값의 관계인 1시차 상관은 **계열상관**이라고도 부릅니다. 이 상관을 이용하려고 첫 열을 반응변수, 나머지 열을 설명변수로 만드는 수식 함수를 작성하겠습니다.

In [55]:
def make_formula(df):
    """Make a Patsy formula from column names."""
    y = df.columns[0]
    xs = " + ".join(df.columns[1:])
    return f"{y} ~ {xs}"

앞선 세 값으로 다음 차분을 예측하는 선형 모형의 결과입니다.

In [56]:
formula = make_formula(df_ar)
results_ar = smf.ols(formula=formula, data=df_ar).fit()
display_summary(results_ar)

`predict`로 과거 값의 적합값을 구해 실제 차분과 비교합니다.

In [57]:
pred_ar = results_ar.predict(df_ar)
pred_ar.plot(label="predictions", **pred_options)
diff.plot(label="differences", **actual_options)
decorate(ylabel="GWh")

잘 맞는 구간도 있지만 $R^2$는 약 0.319로 개선할 여지가 있습니다.

In [58]:
resid_ar = (diff - pred_ar).dropna()
R2 = 1 - resid_ar.var() / diff.var()
R2

첫 모형의 잔차를 구하고 다른 모형으로 그 잔차를 예측할 수 있습니다. 앞의 자기회귀에 오차 조정을 더하는 방법입니다.

## 이동평균 성분

2019년 6월에 2020년 6월을 예측한다면 지난해 같은 달의 반복에서 시작할 수 있습니다. 2020년 5월에 예측을 갱신한다면 최근 세 달의 차분과 앞 절의 자기회귀를 활용할 수 있습니다. 최근 예측들이 계속 낮았다는 사실까지 알게 되면 다음 예측도 위로 조정할 수 있겠습니다. 최근 오차가 다음 오차에 관한 정보를 줄 수 있다는 가정입니다.

이를 보기 위해 첫 자기회귀의 잔차와 1·6개월 늦춘 잔차를 같은 표에 담습니다.

In [59]:
df_ma = pd.DataFrame({"resid": resid_ar})

for lag in [1, 6]:
    df_ma[f"lag{lag}"] = resid_ar.shift(lag)

df_ma = df_ma.dropna()

과거 잔차로 다음 잔차를 설명하는 모형에도 `ols`를 적용합니다. 원문에서는 이를 이동평균 성분을 설명하는 두 단계 접근으로 사용합니다. 예측 변동을 줄이는 효과가 평활 이동평균과 비슷하다는 뜻에서 쓰이는 이름이지만, 단순히 관측값 몇 개의 평균을 낸다는 뜻은 아닙니다. 적합 결과를 보겠습니다.

In [60]:
formula = make_formula(df_ma)
results_ma = smf.ols(formula=formula, data=df_ma).fit()
display_summary(results_ma)

$R^2$는 작아 큰 개선을 기대하기 어렵지만, 6개월 시차 계수의 p값은 작습니다. 잔차의 과거예측을 구합니다.

In [61]:
pred_ma = results_ma.predict(df_ma)

첫 모형의 차분 예측에 두 번째 모형의 잔차 조정을 더합니다.

In [62]:
pred_diff = pred_ar + pred_ma

두 모형을 합친 $R^2$는 약 0.332로, 조정 전의 약 0.319보다 조금 큽니다.

In [63]:
resid_ma = (diff - pred_diff).dropna()
R2 = 1 - resid_ma.var() / diff.var()
R2

예측한 전년동월 차분으로 원래 발전량을 복원하겠습니다.

## 자기회귀의 과거예측

원래 날짜에 맞춰 차분 예측을 `Series`에 담습니다.

In [64]:
pred_diff = pd.Series(pred_diff, index=nuclear.index)

처음 21개 값이 `NaN`입니다.

In [65]:
n_missing = pred_diff.isna().sum()
n_missing

전년동월 차분을 만들며 12개월, 첫 자기회귀에서 3개월, 잔차 모형에서 6개월을 늦췄으므로 합계 21개가 앞에서 빠집니다. 복원을 시작하려고 원자료의 처음 21개를 복사합니다.

In [66]:
pred_series = pd.Series(index=nuclear.index, dtype=float)
pred_series.iloc[:n_missing] = nuclear.iloc[:n_missing]

인덱스 21, 즉 22번째 원소부터 지난해 같은 달 값과 예측 차분을 더해 채웁니다.

In [67]:
for i in range(n_missing, len(pred_series)):
    pred_series.iloc[i] = pred_series.iloc[i - 12] + pred_diff.iloc[i]

처음 복사한 21개는 다시 `NaN`으로 바꿉니다. 그대로 옮긴 값들을 정확히 예측한 것처럼 평가하지 않기 위해서입니다.

In [68]:
pred_series[:n_missing] = np.nan

복원한 과거예측과 원자료를 비교합니다.

In [69]:
pred_series.plot(label="predicted", **pred_options)
nuclear.plot(label="actual", **actual_options)
decorate(ylabel="GWh")

비교적 잘 맞고 $R^2$는 약 0.86입니다.

In [70]:
resid = (nuclear - pred_series).dropna()
R2 = 1 - resid.var() / nuclear.var()
R2

이 과정을 SARIMA라는 모형군과 연결할 수 있습니다. ARIMA 이름의 각 부분은 다음을 가리킵니다.

- **S**(seasonal): 한 계절 주기만큼 떨어진 값들의 차분을 사용합니다.
- **AR**(autoregression): 과거 차분들의 관계로 다음 값을 설명합니다.
- **I**(integrated): 차분 예측을 누적해 원래 수준으로 되돌립니다.
- **MA**(moving average): 과거 예측 오차를 반영하는 성분입니다.

> 번역자 주: 위의 두 단계 OLS는 AR·오차 보정의 생각을 설명하는 구성입니다. 혁신 오차를 함께 추정하는 표준 SARIMA의 우도 적합과 완전히 같은 계산은 아닙니다. 앞 절의 평활용 `rolling().mean()`과 ARIMA의 MA 성분도 구별하세요.

## ARIMA

statsmodels의 `tsa`는 time series analysis의 줄임말입니다. `ARIMA`로 모형을 적합하고 예측할 수 있습니다. 앞서 살펴본 구조를 `order`와 `seasonal_order`라는 두 튜플로 지정하겠습니다.

In [71]:
order = ([1, 2, 3], 0, [1, 6])

`order`는 AR에 포함할 시차, 비계절 차분 횟수, MA에 포함할 시차를 나타냅니다. 여기서는 AR 시차 1~3, 비계절 차분 0회, MA 시차 1과 6을 선택합니다. 다음은 계절 성분의 지정입니다.

In [72]:
seasonal_order = (0, 1, 0, 12)

`seasonal_order`의 첫째와 셋째가 0이므로 계절 AR·MA는 없습니다. 둘째 1은 계절 차분 1회, 마지막 12는 12개월 주기입니다. `ARIMA`를 적합합니다.

In [73]:
import statsmodels.tsa.api as tsa

model = tsa.ARIMA(nuclear, order=order, seasonal_order=seasonal_order)
results_arima = model.fit()
display_summary(results_arima)

결과에는 AR 계수 세 개, MA 계수 두 개, 혁신 오차 분산 `sigma2`가 있습니다. `fittedvalues`에서 적합값을 꺼내되, 초기 조건에 민감한 맨 앞 부분은 제외합니다.

In [74]:
fittedvalues = results_arima.fittedvalues[n_missing:]

직접 구성한 결과와 비슷하지만 정확히 같지는 않습니다. 초기 조건 처리 등 적합 방법의 차이가 있습니다.

In [75]:
fittedvalues.plot(label="ARIMA model", **pred_options)
nuclear.plot(label="actual", **actual_options)
decorate(ylabel="GWh")

$R^2$도 비슷하지만 일치하지는 않습니다.

In [76]:
resid = fittedvalues - nuclear
R2 = 1 - resid.var() / nuclear.var()
R2

`ARIMA`를 쓰면 다른 차수도 쉽게 실험할 수 있습니다.

`order`와 `seasonal_order`를 바꾸어 더 높은 적합 $R^2$를 찾으세요. 미래 예측에 쓸 모형을 고를 때는 훈련 적합도만이 아니라 별도의 시간순 평가 구간도 사용해야 합니다.

## ARIMA로 예측하기

적합 결과의 `get_forecast`로 미래를 예측할 수 있습니다. 다시 훈련·평가 자료를 나누고 훈련 구간에만 같은 모형을 맞춥니다.

In [77]:
training, test = split_series(nuclear)
model = tsa.ARIMA(training, order=order, seasonal_order=seasonal_order)
results_training = model.fit()

평가 구간 길이만큼 예측합니다.

In [78]:
forecast = results_training.get_forecast(steps=len(test))

반환 객체의 `predicted_mean`에 예측값이 들어 있고, `conf_int()`로 예측구간을 구할 수 있습니다.

> 번역자 주: 원문 설명의 `forecast_mean`은 아래 코드의 `predicted_mean`에 맞춰 고쳤습니다. 여기서 그리는 것은 미래 관측값의 변동을 반영한 모형 기반 **예측구간**입니다. 회귀선의 평균에 대한 신뢰구간과 구분해야 합니다. 모형 선택·모수 추정·구조 변화의 모든 불확실성이 완전히 반영되는 것은 아닙니다.

반환 객체는 statsmodels의 [`get_forecast` 문서](https://www.statsmodels.org/stable/generated/statsmodels.tsa.statespace.sarimax.SARIMAXResults.get_forecast.html)를 참고하세요.

In [79]:
forecast_mean = forecast.predicted_mean
forecast_ci = forecast.conf_int()
forecast_ci.columns = ["lower", "upper"]

예측과 구간을 실제 시계열 위에 그립니다.

In [80]:
plt.fill_between(
    forecast_ci.index,
    forecast_ci.lower,
    forecast_ci.upper,
    lw=0,
    color="gray",
    alpha=0.2,
)
plt.plot(forecast_mean.index, forecast_mean, label="forecast", **pred_options)
plt.plot(test.index, test, label="actual", **actual_options)
decorate(ylabel="GWh")

이 예에서는 실제 값 대부분이 예측구간 안에 있습니다. MAPE를 계산하겠습니다.

In [81]:
MAPE(forecast_mean, test)

원문의 MAPE는 약 3.38%로 계절 분해의 약 3.81%보다 조금 작습니다. ARIMA는 더 다양한 구조를 표현할 수 있지만 이 시계열의 자기상관이 아주 강하지 않아 이득은 크지 않습니다.

## 용어 정리

- **시계열**: 각 관측값이 특정 시점에 연결된 자료.
- **계절 분해**: 시계열을 장기 추세·반복되는 계절 성분·잔차로 나누는 방법.
- **훈련 시계열**: 모형 적합에 사용하는 과거 구간.
- **평가 시계열**: 적합에 쓰지 않고 예측 정확도를 확인하는 구간.
- **과거예측**: 이미 관측된 과거 시점에 대한 모형의 예측 또는 적합값.
- **창**: 이동평균 계산에 사용하는 연속 관측값 묶음.
- **이동평균**: 겹치는 창 안의 평균으로 변동을 매끄럽게 만든 시계열. ARIMA의 MA 성분과는 다른 용법입니다.
- **계열상관**: 인접한 관측값들 사이의 상관.
- **자기상관**: 시계열과 시차를 준 자기 자신 사이의 상관.
- **시차**: 서로 다른 시점의 관측값을 비교할 때 두 시점 사이의 간격.

## 연습문제

### 연습문제 12.1

미국의 월평균 지표 부근 기온을 계절 분해해 보세요. Our World in Data의 자료는 1950~2024년 여러 나라에서 육지·바다·내륙 수면을 포함한 지상 약 2m의 공기 온도를 섭씨로 나타냅니다. 자료를 가져오는 코드는 노트북에 있습니다.

In [82]:
# The following cell downloads data prepared by Our World in Data,
# which I Downloaded September 18, 2024
# from https://ourworldindata.org/grapher/average-monthly-surface-temperature

# Based on modified data from Copernicus Climate Change Service information (2019)
# with "major processing" by Our World in Data

In [83]:
filename = "monthly-average-surface-temperatures-by-year.csv"
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/" + filename)

기온 자료를 다음과 같이 읽습니다.

In [84]:
temp = pd.read_csv("monthly-average-surface-temperatures-by-year.csv")

In [85]:
temp.head()

2001년부터 마지막 관측까지의 미국 자료를 골라 `Series`로 만듭니다.

In [86]:
temp_us = temp.query("Code == 'USA'")
columns = [str(year) for year in range(2000, 2025)]
temp_series = temp_us.loc[:, columns].transpose().stack().dropna()
temp_series.index = pd.date_range(start="2000-01", periods=len(temp_series), freq="ME")

선택한 시계열을 그립니다.

In [87]:
temp_series.plot(label="monthly average", **actual_options)
decorate(ylabel="Surface temperature (degC)")

강한 계절 패턴이 보입니다. 12개월 주기의 가법 분해를 하고 추세에 직선을 맞추세요. 이 기간 평균적인 연간 기온 증가는 얼마인가요? 다른 기간이나 나라에서도 반복해 볼 수 있습니다.

In [88]:
# Solution goes here

In [89]:
# Solution goes here

In [90]:
# Solution goes here

In [91]:
# Solution goes here

In [92]:
# Solution goes here

In [93]:
# Solution goes here

In [94]:
# Solution goes here

In [95]:
# Solution goes here

In [96]:
# Solution goes here

### 연습문제 12.2

앞에서는 소규모 태양광의 2014~2019년 자료로 2019~2024년을 예측했습니다. 이번에는 대규모 태양광 발전량으로 같은 분석을 하세요.

In [97]:
util_solar = elec["United States : all utility-scale solar"].dropna()
util_solar = util_solar[util_solar.index.year >= 2014]
util_solar.plot(**actual_options)
decorate(ylabel="GWh")

`split_series`로 훈련·평가 구간을 나누고 12개월 주기의 승법 분해를 하세요. 추세에 일차 또는 이차 모형을 맞추어 계절 성분을 포함한 5년 예측을 만드세요. 실제 값과 비교하고 MAPE를 구하세요.

In [98]:
# Solution goes here

In [99]:
# Solution goes here

In [100]:
# Solution goes here

In [101]:
# Solution goes here

In [102]:
# Solution goes here

In [103]:
# Solution goes here

In [104]:
# Solution goes here

In [105]:
# Solution goes here

In [106]:
# Solution goes here

### 연습문제 12.3

2001~2024년 미국 수력 발전량에는 ARIMA가 얼마나 잘 맞을까요? 다음 시계열을 사용하세요.

In [107]:
hydro = elec["United States : conventional hydroelectric"]
hydro.plot(**actual_options)
decorate(ylabel="GWh")

12개월 주기의 SARIMA를 적합하고 AR·MA 시차를 바꾸어 적합도를 비교하세요. 5년 예측과 예측구간을 그리세요. 선택한 시차에 따라 첫 12~24개 적합값이 불안정할 수 있으므로 그래프와 $R^2$ 계산에서 제외할 수 있습니다. 모형을 고른 평가 자료를 최종 성능 확인에 재사용하는 한계에도 주의하세요.

In [108]:
# Solution goes here

In [109]:
# Solution goes here

In [110]:
# Solution goes here

In [111]:
# Solution goes here

In [112]:
# Solution goes here

In [113]:
# Solution goes here

[Think Stats: Exploratory Data Analysis in Python, 3rd Edition](https://allendowney.github.io/ThinkStats/index.html)

Copyright 2024 [Allen B. Downey](https://allendowney.com)

코드 라이선스: [MIT](https://mit-license.org/)

본문·그림과 한국어 번역 라이선스: [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

In [114]:
# Solution goes here